## Task 4: Tversky Loss

**Goal:** Use Tversky Loss as an alternative to cross-entropy on the same imbalanced dataset. Tversky Loss allows asymmetric penalisation of false positives and false negatives.

1. Implement `TverskyLoss`:

```python
class TverskyLoss(nn.Module):
    """
    TI   = (TP + smooth) / (TP + alpha*FP + beta*FN + smooth)
    Loss = 1 - TI

    Special cases:
      alpha=0.5, beta=0.5  →  Dice Loss
      alpha=0.0, beta=1.0  →  pure Recall loss

    For imbalanced data, beta > alpha penalises FN more heavily → higher recall.
    """
    def __init__(self, alpha=0.5, beta=0.5, smooth=1e-6, from_logits=True):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta
        self.smooth = smooth
        self.from_logits = from_logits

    def forward(self, logits, targets):
        if self.from_logits:
            probs = torch.sigmoid(logits.squeeze(1) if logits.dim() == 2 else logits)
        else:
            probs = logits.squeeze(1)
        targets = targets.float()
        tp = (probs * targets).sum()
        fp = (probs * (1.0 - targets)).sum()
        fn = ((1.0 - probs) * targets).sum()
        tversky_index = (tp + self.smooth) / (
            tp + self.alpha * fp + self.beta * fn + self.smooth
        )
        return 1.0 - tversky_index
```

2. Train and evaluate analogously to Task 2, for several `(alpha, beta)` configurations:

| alpha | beta | effect                         |
|-------|------|--------------------------------|
| 0.5   | 0.5  | Dice Loss (symmetric)          |
| 0.3   | 0.7  | recall-biased                  |
| 0.1   | 0.9  | strong recall (penalise FN)    |

**Assignment:** Implement and train a classifier on a strongly imbalanced dataset — compare classification quality for BCE and Tversky Loss.


In [1]:
IMBALANCE_RATIO = 0.95

BATCH_SIZE   = 128
EPOCHS       = 1000
LR           = 1e-3
WEIGHT_DECAY = 1e-4

In [2]:
from utils import make_imbalanced_dataset, BinaryClassifierMLP, make_loaders
import torch

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model = BinaryClassifierMLP().to(device)

X, y = make_imbalanced_dataset(imbalance_ratio=IMBALANCE_RATIO)
print(f"Class 0: {(y == 0).sum():.0f}  Class 1: {(y == 1).sum():.0f}")

train_loader, val_loader, test_loader = make_loaders(X, y, batch_size=BATCH_SIZE, weighted_sampler=True, squeeze_y=True)

Class 0: 11288  Class 1: 712


In [3]:
from utils import train_baseline, TverskyLoss
import torch.nn as nn

TVERSKY_CONFIGS = [
    (0.5, 0.5),  # Dice Loss (symmetric)
    (0.3, 0.7),  # recall-biased
    (0.1, 0.9),  # strong recall (penalise FN)
]

print("Training with BCE: ")
model_bce = train_baseline(BinaryClassifierMLP().to(device), train_loader,
                           val_loader,criterion=nn.BCEWithLogitsLoss(),device=device,
                           weight_decay=WEIGHT_DECAY, epochs=EPOCHS, lr=LR
                           )

tversky_models = {}
for alpha, beta in TVERSKY_CONFIGS:
    print(f"\nTraining with TverskyLoss(alpha={alpha}, beta={beta}): ")
    tversky_models[f"Tversky(α={alpha},β={beta})"] = (
        train_baseline(BinaryClassifierMLP().to(device), train_loader,
                    val_loader, criterion=TverskyLoss(alpha=alpha, beta=beta),
                    device=device, weight_decay=WEIGHT_DECAY, epochs=EPOCHS, lr=LR
                       ))

Training with BCE: 
Epoch 100/1000  train=0.0486  val=0.1760
Epoch 200/1000  train=0.0311  val=0.1920
Epoch 300/1000  train=0.0232  val=0.2267
Epoch 400/1000  train=0.0218  val=0.2337
Epoch 500/1000  train=0.0195  val=0.2484
Epoch 600/1000  train=0.0165  val=0.2427
Epoch 700/1000  train=0.0113  val=0.2683
Epoch 800/1000  train=0.0175  val=0.2688
Epoch 900/1000  train=0.0130  val=0.2752
Epoch 1000/1000  train=0.0127  val=0.2700

Training with TverskyLoss(alpha=0.5, beta=0.5): 
Epoch 100/1000  train=0.0574  val=0.3061
Epoch 200/1000  train=0.0535  val=0.2833
Epoch 300/1000  train=0.0441  val=0.2795
Epoch 400/1000  train=0.0475  val=0.2837
Epoch 500/1000  train=0.0448  val=0.2738
Epoch 600/1000  train=0.0491  val=0.2618
Epoch 700/1000  train=0.0432  val=0.2716
Epoch 800/1000  train=0.0432  val=0.2743
Epoch 900/1000  train=0.0425  val=0.2697
Epoch 1000/1000  train=0.0456  val=0.2681

Training with TverskyLoss(alpha=0.3, beta=0.7): 
Epoch 100/1000  train=0.0451  val=0.3300
Epoch 200/1000  t

In [4]:
from utils import get_probs, compute_clf_metrics

y_true, probs_bce = get_probs(model_bce, test_loader, device=device)
m_bce = compute_clf_metrics(y_true, probs_bce)

results = {"BCE": m_bce}
for name, model in tversky_models.items():
    y_true, probs = get_probs(model, test_loader, device=device)
    results[name] = compute_clf_metrics(y_true, probs)

print(f"\n{'Metric':<12}", end="")
for name in results:
    print(f" {name:>22}", end="")
print()
print("=" * (12 + 23 * len(results)))
for key in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]:
    print(f"{key:<12}", end="")
    for m in results.values():
        print(f" {m[key]:>22.4f}", end="")
    print()
print("=" * (12 + 23 * len(results)))
print()
for name, m in results.items():
    print(f"{name} — TP={m['tp']}  FP={m['fp']}  TN={m['tn']}  FN={m['fn']}")


Metric                          BCE   Tversky(α=0.5,β=0.5)   Tversky(α=0.3,β=0.7)   Tversky(α=0.1,β=0.9)
accuracy                     0.9558                 0.9742                 0.9621                 0.9483
precision                    0.5864                 0.8158                 0.6525                 0.5272
recall                       0.7090                 0.6940                 0.6866                 0.7239
f1                           0.6419                 0.7500                 0.6691                 0.6101
roc_auc                      0.8888                 0.9090                 0.9000                 0.8925
pr_auc                       0.7398                 0.7548                 0.7426                 0.6896

BCE — TP=95  FP=67  TN=2199  FN=39
Tversky(α=0.5,β=0.5) — TP=93  FP=21  TN=2245  FN=41
Tversky(α=0.3,β=0.7) — TP=92  FP=49  TN=2217  FN=42
Tversky(α=0.1,β=0.9) — TP=97  FP=87  TN=2179  FN=37


BCE (Binary Cross-Entropy) is a standard binary classification loss function.

Tversky Loss operates directly on TP, FP and FN counts instead of per-example probabilities. It computes a Tversky Index TI = TP / (TP + alpha*FP + beta*FN) and the loss is 1 - TI. Parameters alpha and beta control how much the model is penalised for false positives and false negatives respectively. Setting beta > alpha makes the model focus on reducing FN at the cost of more FP, which directly increases recall. Special case: alpha=0.5, beta=0.5 is equivalent to Dice Loss.

BCE — balanced or slightly imbalanced data
Tversky — conscious control of precision/recall tradeoff, e.g. in medicine where FN (missed disease) is much worse than FP